# 5. Inverse PINNs, Identifiability, and Held-Out Validation

**Owner:** @rohanagrawal28 (WP05)

In this section, we estimate unknown parameters of strong-Allee harvesting models from discrete, noisy population observations.
We investigate two central questions:
1. **Structural Identifiability:** Are model parameters uniquely determined by ideal observation trajectories?
2. **Inference & Generalization:** How accurately can parameters and held-out future states be predicted under measurement noise using conventional ODE least squares versus Physics-Informed Neural Networks (PINNs)?

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Automatic Colab & Local path detection
if 'google.colab' in sys.modules:
    for candidate in [Path.cwd(), Path('/content'), Path('/content/Rohan_RK45_Data_and_Code'), Path('/content/mm-allee-harvesting')]:
        if (candidate / 'src/mm_allee').exists():
            if str(candidate / 'src') not in sys.path:
                sys.path.insert(0, str(candidate / 'src'))
            os.chdir(candidate)
            break
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'src/mm_allee').exists()), None)
    if ROOT is not None and str(ROOT / 'src') not in sys.path:
        sys.path.insert(0, str(ROOT / 'src'))

from mm_allee.models import Parameters, dimensional_rhs
from mm_allee.solver import solve_trajectory
from mm_allee.data import generate_noisy_observations, validate_splits, validate_time_windows
from mm_allee.inference import fit_ode_least_squares
from mm_allee.metrics import errors
from mm_allee.validation import compute_parameter_metrics

print('Loaded shared modules successfully.')

## 5.1 Model B Structural Nonidentifiability and Counterexample

Under Model B, harvesting is proportional to population density ($h N$):
$$\frac{dN}{dt} = r N \left(1 - \frac{N}{K}\right)\left(\frac{N}{A} - 1\right) - h N$$

Expanding the cubic polynomial:
$$\frac{dN}{dt} = -(r + h) N + r\left(\frac{1}{A} + \frac{1}{K}\right) N^2 - \frac{r}{AK} N^3$$

Any trajectory observed at a single constant effort $h$ reveals at most **three** composite polynomial combinations:
$$c_1 = -(r + h), \quad c_2 = r\left(\frac{1}{A} + \frac{1}{K}\right) = \frac{r(A+K)}{AK}, \quad c_3 = -\frac{r}{AK}$$

Because there are 4 unknown physical parameters $(r, A, K, h)$ but only 3 observable coefficients, the system is **structurally nonidentifiable** from data at a single fixed effort.

In [ ]:
# Demonstrate the exact two-parameter-set counterexample:
p1 = Parameters(r=1.0, A=0.2, K=1.0, h=0.5)
p2 = Parameters(r=1.35, A=0.3, K=0.9, h=0.15)

c1_1, c2_1, c3_1 = -(p1.r + p1.h), p1.r * (1.0 / p1.A + 1.0 / p1.K), -p1.r / (p1.A * p1.K)
c1_2, c2_2, c3_2 = -(p2.r + p2.h), p2.r * (1.0 / p2.A + 1.0 / p2.K), -p2.r / (p2.A * p2.K)

print(f'Parameter Set 1: c1={c1_1:.2f}, c2={c2_1:.2f}, c3={c3_1:.2f}')
print(f'Parameter Set 2: c1={c1_2:.2f}, c2={c2_2:.2f}, c3={c3_2:.2f}')
assert np.isclose(c1_1, c1_2) and np.isclose(c2_1, c2_2) and np.isclose(c3_1, c3_2)

# Solve trajectories from three distinct initial states for both parameter sets
d1, d2 = p1.dimensionless(), p2.dimensionless()
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for n0, color in zip([0.4, 0.7, 1.1], colors, strict=True):
    traj1 = solve_trajectory('B', a=d1['a'], harvest=d1['e'], x0=n0 / p1.K, tau_end=p1.r * 4.0, n_points=81)
    traj2 = solve_trajectory('B', a=d2['a'], harvest=d2['e'], x0=n0 / p2.K, tau_end=p2.r * 4.0, n_points=81)
    t1, y1 = traj1.tau / p1.r, traj1.x * p1.K
    t2, y2 = traj2.tau / p2.r, traj2.x * p2.K
    ax.plot(t1, y1, label=f'Set 1 (N0={n0})', color=color, lw=2)
    ax.plot(t2, y2, '--', label=f'Set 2 (N0={n0})', color='black', alpha=0.7)

ax.set_title('Model B Identifiability Counterexample: Identical Trajectories from Different Parameters')
ax.set_xlabel('Time t')
ax.set_ylabel('Population N(t)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5.2 Observations, Noise Generation, and Frozen 60/20/20 Splits

We construct noisy observations according to the contract:
$$N_{\text{obs}}(t_i) = N_{\text{clean}}(t_i) + \sigma \cdot N_{\text{ref}} \cdot \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, 1)$$
with fixed external scale $N_{\text{ref}} = 100.0$. Observations are never silently clipped.

Splits are strictly partitioned into contiguous time windows:
- **Train (60%):** used exclusively for fitting and data loss.
- **Validation (20%):** used for early stopping and model selection.
- **Held-out Test (20%):** completely frozen until final post-training evaluation.

In [ ]:
p_true = Parameters(r=0.7, K=100.0, A=20.0, H=10.0)
d_true = p_true.dimensionless()
ref_traj = solve_trajectory('A', a=d_true['a'], harvest=d_true['q'], x0=45.0 / 100.0, tau_end=0.7 * 8.0, n_points=101)
times = ref_traj.tau / 0.7
clean_pops = ref_traj.x * 100.0

obs_5pct = generate_noisy_observations(
    times, clean_pops, trajectory_id='a_regular', model='A', noise_sigma=0.05, noise_seed=11, n_ref=100.0
)

idx_tr, idx_va, idx_te = obs_5pct['train_indices'], obs_5pct['val_indices'], obs_5pct['test_indices']
print(f'Train points: {len(idx_tr)} (0 to {times[idx_tr[-1]]:.2f})')
print(f'Val points:   {len(idx_va)} ({times[idx_va[0]]:.2f} to {times[idx_va[-1]]:.2f})')
print(f'Test points:  {len(idx_te)} ({times[idx_te[0]]:.2f} to {times[idx_te[-1]]:.2f})')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axvspan(0, times[idx_tr[-1]], color='blue', alpha=0.1, label='Train Split (60%)')
ax.axvspan(times[idx_tr[-1]], times[idx_va[-1]], color='green', alpha=0.1, label='Val Split (20%)')
ax.axvspan(times[idx_va[-1]], times[-1], color='red', alpha=0.1, label='Held-out Test (20%)')
ax.plot(times, clean_pops, 'k-', lw=2, label='Clean Reference')
ax.scatter(times, obs_5pct['observed_population'], color='darkblue', s=20, alpha=0.6, label='Noisy Observations (5% noise)')
ax.set_title('Model A: Observation Splits and Noise Structure')
ax.set_xlabel('Time t')
ax.set_ylabel('Population N(t)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.3 Conventional Nonlinear Least-Squares ODE Baseline

We fit unknown parameters using SciPy's Trust Region Reflective solver on training observations only, with known $K=100.0$ and bounds $r > 0, 0 < A < K, H \ge 0$.

In [ ]:
fit_res = fit_ode_least_squares(
    'A',
    observations=obs_5pct,
    known_conditions={'known_K': 100.0, 'known_initial_population': 45.0},
    fit_params=['r', 'A', 'H'],
    initial_guess={'r': 1.0, 'A': 30.0, 'H': 5.0},
    n_ref=100.0,
)

print(f'ODE Fit Status: {fit_res.status} in {fit_res.wall_seconds:.3f}s')
print('Estimated Parameters:')
for p in ['r', 'A', 'H']:
    true_v = getattr(p_true, p)
    est_v = fit_res.estimated_parameters[p]
    rel_err = abs(est_v - true_v) / true_v * 100
    print(f'  {p}: True = {true_v:.2f}, Estimated = {est_v:.4f} (Error: {rel_err:.2f}%)')

print('Held-out Test Performance:')
print(f'  RMSE: {fit_res.test_metrics["rmse"]:.4f}, Relative L2: {fit_res.test_metrics["relative_l2"]:.4f}')

## 5.4 Inverse PINN with Differentiable Constraints

We optimize network weights and physical parameters jointly, preserving gradients via:
$$r = \mathrm{softplus}(\tilde{r}) > 0, \quad A = K \cdot \sigma(\tilde{A}) \in (0, K), \quad H = \mathrm{softplus}(\tilde{H}) \ge 0$$
Training loss is evaluated **strictly on training observations** with validation early stopping.

In [ ]:
try:
    import torch
    from mm_allee.pinn.network import PINN
    from mm_allee.pinn.inverse import TrainableParameters, InversePINNTrainer

    net = PINN(hidden_layers=[32, 32], activation='tanh', t_ref=1.0, n_ref=100.0)
    tp = TrainableParameters('A', fit_params=['r', 'A', 'H'], known_conditions={'known_K': 100.0})
    trainer = InversePINNTrainer(net, tp, lr_net=1e-3, lr_params=1e-2, n_ref=100.0)
    pinn_res = trainer.fit(obs_5pct, known_conditions={'known_initial_population': 45.0}, epochs=200, patience=30, seed=11)

    print(f'PINN Completed: {pinn_res.status} in {pinn_res.wall_seconds:.2f}s (Final Epoch: {pinn_res.final_epoch})')
    for p in ['r', 'A', 'H']:
        true_v = getattr(p_true, p)
        est_v = pinn_res.estimated_parameters[p]
        rel_err = abs(est_v - true_v) / true_v * 100
        print(f'  {p}: True = {true_v:.2f}, PINN Est = {est_v:.4f} (Error: {rel_err:.2f}%)')
except ImportError:
    print('PyTorch is not installed in this environment; skipping PINN training. Full code is implemented in mm_allee.pinn.')

## 5.5 Key Scientific Takeaways

1. **Parameter Error $\ne$ Trajectory Error:** PINNs can fit trajectory data closely while settling on incorrect parameter estimates due to overparameterization.
2. **Classical ODE Baselines are Strong:** For scalar ODEs with low-dimensional parameter spaces, SciPy nonlinear least squares is faster and achieves superior parameter accuracy.
3. **Structural Identifiability Precedes Optimization:** Model B with unknown effort $h$ is mathematically ambiguous; no optimization algorithm can break this ambiguity without additional independent information (e.g. known $K$ or multiple effort levels).